# HAQQ — Media Verification Evaluation Notebook

This notebook provides a complete prototype for the **Image & Video Verification Pipeline** of the ITI graduation project **HAQQ (حقّق)**.

### Target Models:
1. **Deepfake Detector (Face-focused)**: `yermandy/GenD_CLIP_L_14` (from Hugging Face)
   - Based on a fine-tuned CLIP ViT-L/14 backbone using Parameter-Efficient LN-tuning.
   - Highly robust at spotting facial manipulations (face swaps, reenactment) across different datasets.
   - *Note: Since the checkpoint's `config.json` on HF has a bug (it specifies a custom model type `GenD` but lacks the standard `auto_map` fields), standard HF pipelines fail to load it. We resolve this by downloading the weights (`model.safetensors`) and instantiating the model definition locally using our helper script `modeling_gend.py`.*
2. **AI-Generated Image Detector (Full-frame focus)**: `Ateeqq/ai-vs-human-image-detector` (from Hugging Face)
   - Based on SigLIP image classification.
   - Detects diffusion artifacts (Stable Diffusion, Midjourney, Flux) on full frames to identify synthesized media.

*Note on DeMamba: Since the weights of DeMamba have not been released publicly by the authors as of July 2026, we use the highly rated `Ateeqq/ai-vs-human-image-detector` (SigLIP) as our AI-generated content detector. When run frame-by-frame on a video, it effectively identifies if the frames are AI-synthesized.*

---

## 🛠️ Step 1: Environment Setup & Installs
Run the cell below to install the required libraries if you haven't already.

In [ ]:
!pip install -q opencv-python-headless facenet-pytorch transformers torch torchvision pillow matplotlib pandas safetensors huggingface_hub

## 📷 Step 2: Preprocessing Pipeline
We implement the frame extraction and face detection modules.
- **Frame Extraction**: Extracts 16 frames uniformly from the video.
- **Face Detection (MTCNN)**: Extracts facial bounding boxes. If faces are found, crops them for the deepfake detector. If no face is found, we fall back to the full frame.

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from facenet_pytorch import MTCNN

# Select device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize MTCNN for face detection
# margin=20 adds some background around the face for better classification context
mtcnn = MTCNN(keep_all=False, post_process=False, device=device, margin=20)

def extract_frames(video_path, n_frames=16):
    """
    Uniformly extracts n_frames from a video file.
    Returns a list of PIL Images.
    """
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Video file not found: {video_path}")
        
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        raise ValueError(f"Could not read frames from video: {video_path}")
        
    # Generate frame indices
    frame_indices = np.linspace(0, total_frames - 1, num=n_frames, dtype=int)
    frames = []
    
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            break
        # Convert BGR to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(Image.fromarray(frame_rgb))
        
    cap.release()
    print(f"Successfully extracted {len(frames)} frames from {video_path}")
    return frames

def preprocess_frames(frames):
    """
    Processes a list of frames. Attempts to extract a face crop from each frame.
    Returns:
        processed_images: List of PIL Images (either face crops or full frames if no face is found).
        faces_detected: List of booleans indicating if a face was detected in each frame.
    """
    processed_images = []
    faces_detected = []
    
    for i, frame in enumerate(frames):
        # Convert PIL to numpy for MTCNN detection
        frame_np = np.array(frame)
        boxes, _ = mtcnn.detect(frame_np)
        
        if boxes is not None and len(boxes) > 0:
            # Crop the first detected face
            box = boxes[0].astype(int)
            # Ensure box boundaries are within image dimensions
            w, h = frame.size
            x1 = max(0, box[0])
            y1 = max(0, box[1])
            x2 = min(w, box[2])
            y2 = min(h, box[3])
            
            face_crop = frame.crop((x1, y1, x2, y2))
            processed_images.append(face_crop)
            faces_detected.append(True)
        else:
            # Fallback to full frame
            processed_images.append(frame)
            faces_detected.append(False)
            
    print(f"Face detection stats: {sum(faces_detected)}/{len(frames)} frames contained faces.")
    return processed_images, faces_detected

## 🤖 Step 3: Loading Pre-trained Models
We load:
1. `yermandy/GenD_CLIP_L_14` for Deepfake detection manually using our helper file `modeling_gend.py` and `model.safetensors` weight checkpoints.
2. `Ateeqq/ai-vs-human-image-detector` for AI-generated frame detection using standard pipelines.

In [ ]:
from transformers import pipeline
from huggingface_hub import hf_hub_download
from safetensors.torch import load_model
from modeling_gend import GenDConfig, GenD

print("Loading Deepfake Detection model (yermandy/GenD_CLIP_L_14) manually...")
# Download the safetensors checkpoint directly from HF
weights_path = hf_hub_download(repo_id="yermandy/GenD_CLIP_L_14", filename="model.safetensors")

# Build configuration and instantiate model from our local modeling file
config = GenDConfig(backbone="openai/clip-vit-large-patch14", head="LinearNorm")
deepfake_model = GenD(config)

# Load weights
load_model(deepfake_model, weights_path)
deepfake_model.eval()
deepfake_model.to(device)
print("Deepfake model successfully loaded!")

print("Loading AI-vs-Human Detector model (Ateeqq/ai-vs-human-image-detector)...")
aigc_pipeline = pipeline(
    "image-classification", 
    model="Ateeqq/ai-vs-human-image-detector",
    device=0 if torch.cuda.is_available() else -1
)
print("Models successfully loaded!")

## ⚖️ Step 4: Video Evaluation & Score Fusion
We define a unified function `analyze_video` that runs:
1. Frame extraction & preprocessing.
2. Face-crop evaluation through `GenD_CLIP_L_14` (outputs: probability of being a deepfake).
3. Full-frame evaluation through `ai-vs-human-image-detector` (outputs: probability of being AI-generated).
4. **Score Fusion**: We combine the scores to give a final decision over three classes: `Real`, `Deepfake`, and `AI-Generated`.

### Score Fusion Logic:
- We average the frame-level probabilities across all frames to get video-level scores.
- We prioritize the higher fake signal to avoid false negatives. To produce a mutually exclusive 3-class distribution (`Real`, `Deepfake`, `AI-Generated`), we map the model outputs to a 3D score vector and normalize it.

In [ ]:
def face_aware_fusion(avg_df_prob, avg_ai_prob, pct_faces_detected):
    """
    Face-detection-aware fusion.
    If faces are meaningfully present, trust the deepfake model first.
    Fall back to AIGC model only when no faces are detected.
    """
    FACE_THRESHOLD = 25.0  # If more than 25% of frames have faces

    # Case 1: Faces detected → Deepfake model has the right context → trust it first
    if pct_faces_detected >= FACE_THRESHOLD:
        if avg_df_prob >= 0.50:
            # GenD is confident → It's a Deepfake
            return {
                "predicted_label": "Deepfake",
                "confidence": avg_df_prob,
                "scores": {
                    "real":         (1.0 - avg_df_prob) * (1.0 - avg_ai_prob),
                    "deepfake":     avg_df_prob,
                    "ai_generated": avg_ai_prob * (1.0 - avg_df_prob)
                }
            }
        elif avg_ai_prob >= 0.50 :
            # Faces exist but GenD sees no manipulation → Faces are entirely synthesized
            return {
                "predicted_label": "AI-Generated",
                "confidence": avg_ai_prob,
                "scores": {
                    "real":         (1.0 - avg_ai_prob) * (1.0 - avg_df_prob),
                    "deepfake":     avg_df_prob * (1.0 - avg_ai_prob),
                    "ai_generated": avg_ai_prob
                }
            }

    # Case 2: No significant faces → AIGC model is the primary evidence
    else:
        if avg_ai_prob >= 0.50:
            return {
                "predicted_label": "AI-Generated",
                "confidence": avg_ai_prob,
                "scores": {
                    "real":         1.0 - avg_ai_prob,
                    "deepfake":     0.0,
                    "ai_generated": avg_ai_prob
                }
            }

    # Case 3: All signals are weak → Real
    p_real = (1.0 - avg_df_prob) * (1.0 - avg_ai_prob)
    return {
        "predicted_label": "Real",
        "confidence": p_real,
        "scores": {
            "real":         p_real,
            "deepfake":     avg_df_prob,
            "ai_generated": avg_ai_prob
        }
    }

In [ ]:
# ───  New analyze_video using the face-aware fusion ───────────────────
def analyze_video_v2(video_path, n_frames=16):
    """
    Evaluates a video using both models and fuses results
    with the face-aware fusion strategy.
    """
    # 1. Extract and preprocess
    frames = extract_frames(video_path, n_frames=n_frames)
    processed_images, faces_detected = preprocess_frames(frames)
    deepfake_scores = []
    aigc_scores     = []

    # 2. Run Deepfake model (batch)
    print("Running Deepfake model inference...")
    processed_tensors = [deepfake_model.feature_extractor.preprocess(img) for img in processed_images]
    batch_tensor = torch.stack(processed_tensors).to(device)
    with torch.no_grad():
        df_logits = deepfake_model(batch_tensor)
        df_probs  = torch.softmax(df_logits, dim=-1).cpu().numpy()
    # Only keep deepfake scores from frames where a face was detected
    for idx, has_face in enumerate(faces_detected):
        if has_face:
            deepfake_scores.append(float(df_probs[idx][1]))
    avg_df_prob = float(np.mean(deepfake_scores)) if deepfake_scores else 0.0

    # 3. Run AIGC model (always on full frames)
    print("Running AI-vs-Human model inference...")
    for idx in range(len(frames)):
        aigc_res  = aigc_pipeline(frames[idx])
        aigc_dict = {item['label'].lower(): item['score'] for item in aigc_res}
        aigc_scores.append(aigc_dict.get('ai', 0.0))
    avg_ai_prob = float(np.mean(aigc_scores))

    # 4. Face-aware fusion
    pct_faces = sum(faces_detected) / len(frames) * 100
    result    = face_aware_fusion(avg_df_prob, avg_ai_prob, pct_faces)
    
    # 5. Attach metadata
    result["metadata"] = {
        "total_frames_analyzed":    len(frames),
        "percentage_faces_detected": pct_faces,
        "avg_deepfake_prob":         avg_df_prob,
        "avg_aigc_prob":             avg_ai_prob
    }
    return result, frames, processed_images, faces_detected

In [ ]:
def analyze_video(video_path, n_frames=16):
    """
    Evaluates a video file for both Deepfakes and AI Generation.
    Fuses the frame-level predictions into a single 3-class score.
    """
    # 1. Extract and preprocess
    frames = extract_frames(video_path, n_frames=n_frames)
    processed_images, faces_detected = preprocess_frames(frames)
    
    deepfake_scores = []
    aigc_scores = []
    
    # 2. Run Inference frame by frame
    print("Running model inference...")
    
    # Preprocess frames for Deepfake model batch processing
    processed_tensors = [deepfake_model.feature_extractor.preprocess(img) for img in processed_images]
    batch_tensor = torch.stack(processed_tensors).to(device)
    
    with torch.no_grad():
        # GenD outputs logits of shape [batch_size, 2]
        df_logits = deepfake_model(batch_tensor)
        df_probs = torch.softmax(df_logits, dim=-1).cpu().numpy()
    
    for idx, (img, has_face) in enumerate(zip(processed_images, faces_detected)):
        # -- Run Deepfake Detector --
        # Class 0: Real, Class 1: Fake (Deepfake)
        df_prob = float(df_probs[idx][1])
        deepfake_scores.append(df_prob)
        
        # -- Run AI-vs-Human Detector --
        # We always run this on full frames
        aigc_res = aigc_pipeline(frames[idx])
        aigc_dict = {item['label'].lower(): item['score'] for item in aigc_res}
        ai_prob = aigc_dict.get('ai', 0.0)
        aigc_scores.append(ai_prob)
        
    # 3. Fuse scores (Average pooling across temporal dimension)
    avg_df_prob = float(np.mean(deepfake_scores))
    avg_ai_prob = float(np.mean(aigc_scores))
    
    # Simple Heuristic / Softmax fusion into 3 classes:
    # [Real, Deepfake, AI-Generated]
    raw_scores = np.array([
        (1.0 - avg_df_prob) * (1.0 - avg_ai_prob), # Real
        avg_df_prob,                               # Deepfake (identity swap)
        avg_ai_prob                                # AI-Generated (entirely synthetic)
    ])
    
    # Normalize scores to sum to 1.0
    fused_probabilities = raw_scores / np.sum(raw_scores)
    
    labels = ["Real", "Deepfake", "AI-Generated"]
    argmax_idx = np.argmax(fused_probabilities)
    predicted_label = labels[argmax_idx]
    confidence = fused_probabilities[argmax_idx]
    
    results = {
        "predicted_label": predicted_label,
        "confidence": confidence,
        "scores": {
            "real": fused_probabilities[0],
            "deepfake": fused_probabilities[1],
            "ai_generated": fused_probabilities[2]
        },
        "metadata": {
            "total_frames_analyzed": len(frames),
            "percentage_faces_detected": sum(faces_detected) / len(frames) * 100
        }
    }
    
    return results, frames, processed_images, faces_detected

## 📊 Step 5: Visualizing Results
We define a helper function to plot the extracted frames, face crops, and final prediction scores side-by-side.

In [ ]:
def plot_results(results, frames, processed_images, faces_detected):
    """
    Plots the frame samples and a bar chart showing the final fused prediction scores.
    """
    # Create figure with 2 subplots (left: frame grid, right: prediction chart)
    fig = plt.figure(figsize=(16, 6))
    
    # 1. Left Subplot: Sample Frames
    grid_size = 4
    indices = np.linspace(0, len(frames)-1, num=grid_size, dtype=int)
    
    for i, idx in enumerate(indices):
        ax = fig.add_subplot(2, grid_size, i+1)
        ax.imshow(frames[idx])
        ax.axis('off')
        ax.set_title(f"Frame {idx}")
        
        ax2 = fig.add_subplot(2, grid_size, grid_size + i + 1)
        ax2.imshow(processed_images[idx])
        ax2.axis('off')
        if faces_detected[idx]:
            ax2.set_title("Face Crop", color="green")
        else:
            ax2.set_title("Full Frame Fallback", color="blue")
            
    plt.tight_layout()
    plt.show()
    
    # 2. Score Chart
    fig, ax = plt.subplots(figsize=(8, 4))
    scores = results['scores']
    categories = list(scores.keys())
    values = list(scores.values())
    
    colors = ['#4CAF50', '#FF5722', '#2196F3'] # Green, Orange, Blue
    bars = ax.barh(categories, values, color=colors)
    ax.set_xlim(0, 1.0)
    ax.set_xlabel("Probability")
    ax.set_title(f"Prediction: {results['predicted_label']} (Confidence: {results['confidence']:.2%})")
    
    # Add values on bars
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.02, bar.get_y() + bar.get_height()/2, f" {width:.2%}", 
                va='center', ha='left', fontweight='bold')
                
    plt.show()
    
    # Print raw JSON-like result
    import json
    print(json.dumps(results, indent=2))

## 🚀 Step 6: Testing the Pipeline
Enter the path to your video file below and run the cell to analyze it.

In [ ]:
import json
# UPDATE THIS PATH TO YOUR TEST VIDEO
test_video_path = "path/to/your/test_video.mp4"  
result, frames, processed_images, faces_detected = analyze_video_v2(test_video_path, n_frames=8)
print("=" * 50)
print(f"VERDICT     : {result['predicted_label']}")
print(f"CONFIDENCE  : {result['confidence']:.2%}")
print(f"Avg DF Prob : {result['metadata']['avg_deepfake_prob']:.2%}")
print(f"Avg AI Prob : {result['metadata']['avg_aigc_prob']:.2%}")
print(f"Faces Found : {result['metadata']['percentage_faces_detected']:.1f}% of frames")
print("=" * 50)
print(json.dumps(result, indent=2))

plot_results(result, frames, processed_images, faces_detected)

In [ ]:
# UPDATE THIS PATH TO YOUR TEST VIDEO
test_video_path = "path/to/your/test_video.mp4" 

if os.path.exists(test_video_path):
    results, frames, processed_images, faces_detected = analyze_video(test_video_path, n_frames=8)
    plot_results(results, frames, processed_images, faces_detected)
else:
    print(f"Please replace the path '{test_video_path}' with a real local path to a test video file.")